# 01 — Test Dataset Loader

Verify that `.mat` datasets load correctly and can be converted to torch tensors.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
from ms_zerogad.data.loader import load_graph_dataset, dataset_info, print_dataset_info
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
import torch

## Load all available datasets

In [ ]:
data_dir = '/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw'
mat_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.mat')])

all_datasets = {}

for fname in mat_files:
    path = os.path.join(data_dir, fname)
    name = fname.replace('.mat', '')
    try:
        A, X, y = load_graph_dataset(path)
        info = dataset_info(A, X, y)
        print_dataset_info(name, info)
        all_datasets[name] = (A, X, y)
    except Exception as e:
        print(f'\n=== {name} ===\n  ERROR: {e}')

## Test conversion to torch tensors

In [ ]:
# Pick a small dataset for testing
test_name = 'Cora' if 'Cora' in all_datasets else list(all_datasets.keys())[0]
A_sp, X_sp, y_np = all_datasets[test_name]

print(f'Testing conversion on {test_name}...\n')

# Convert
A = sparse_to_torch_dense(A_sp)
X = feature_to_torch(X_sp, dense=True)
y = torch.from_numpy(y_np).long()

print(f'A: {A.shape}, dtype={A.dtype}, sym={torch.allclose(A, A.T)}')
print(f'   diagonal sum (self-loops): {A.diagonal().sum().item()}')
print(f'   value range: [{A.min().item()}, {A.max().item()}]')
print(f'X: {X.shape}, dtype={X.dtype}')
print(f'   value range: [{X.min().item():.4f}, {X.max().item():.4f}]')
print(f'y: {y.shape}, dtype={y.dtype}, anomaly count={y.sum().item()}')

## Verify graph properties

In [ ]:
# Check connectivity
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

n_components, labels_cc = connected_components(A_sp, directed=False)
print(f'Connected components: {n_components}')
if n_components > 1:
    component_sizes = []
    for i in range(n_components):
        component_sizes.append((labels_cc == i).sum())
    component_sizes.sort(reverse=True)
    print(f'Top-5 component sizes: {component_sizes[:5]}')

## Check label distribution per dataset

In [ ]:
import numpy as np

print(f'{"Dataset":<15} {"Total":>8} {"Anomalies":>10} {"Ratio":>8}')
print('-' * 45)
for name, (A, X, y) in all_datasets.items():
    n = len(y)
    n_anom = int(y.sum())
    ratio = n_anom / n
    print(f'{name:<15} {n:>8,} {n_anom:>10,} {ratio*100:>7.2f}%')